In [ ]:
# Журнал 
LOG_COLS = ["field", "issue", "scope", "n_rows", "value_before", "action", "value_after", "rationale"]
cleaning_log = []

def add_log(field, issue, scope, n_rows, value_before, action, value_after, rationale):
    cleaning_log.append(dict(zip(LOG_COLS, [field, issue, scope, n_rows, value_before, action, value_after, rationale])))


## 0. Зачитаем данные 

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_PATH = Path("..") / "data" / "raw" / "rent_v28.csv"
INTERIM_PATH = Path("..") / "data" / "interim" / "test.csv"

print(RAW_PATH.resolve(), "| файл найден:", RAW_PATH.exists())

df_naive = pd.read_csv(RAW_PATH, sep=";", encoding="cp1251", dtype={"id": "string"})

print(f"Размер: {df_naive.shape[0]} строк × {df_naive.shape[1]} столбцов")
df_naive.head(3)

Вывод: кодировку необходимо использовать "cp1251", а разделителями являются |
## 1. Минимальная подготовка данных до отделения test
### 1.1 Чтение данных с корректными типами

In [ ]:
READ_KW = dict(sep="|", encoding="cp1251")

STR_COLS = ["listing_id"]
DATE_COLS = ["publication_date"]
CAT_COLS = ["district", "house_type", "elevator", "renovation", "furniture", "appliances", "balcony", "parking", "pets_allowed", "seller_type"]
INT_COLS = ["year_built", "floors_total", "floor", "rooms", "metro_distance_m", "photos_count", "max_tenants", "rent_rub"]
FLOAT_COLS = ["total_area", "living_area", "kitchen_area", "ceiling_height", "distance_to_center_km", "response_time_hours"]

# Читаем всё как текст: в числах есть пробелы и десятичная запятая
df = pd.read_csv(RAW_PATH, sep="|", encoding="cp1251", dtype=str, keep_default_na=False, na_values=[""])
print(f"Размер файла: {df.shape[0]} строк × {df.shape[1]} столбцов")

def to_num(s):
    clean = s.str.replace(r"[\s]", "", regex=True).str.replace(",", ".", regex=False)
    return pd.to_numeric(clean, errors="coerce")

df[STR_COLS] = df[STR_COLS].astype("string")
df[CAT_COLS] = df[CAT_COLS].astype("category")

# Дата: исходный текст сохраняем в s, чтобы найти значения, которые не удалось привести
s = df[DATE_COLS[0]]
df[DATE_COLS[0]] = pd.to_datetime(s, errors="coerce")
bad_dates = s.notna() & df[DATE_COLS[0]].isna()
print(f"{DATE_COLS[0]}: не приведено {bad_dates.sum()}, примеры: {s[bad_dates].unique()[:5]}")

for c in INT_COLS + FLOAT_COLS:
    s = df[c]
    df[c] = to_num(s)
    bad = s.notna() & df[c].isna()
    if bad.any():
        print(f"{c}: не приведено {bad.sum()} знач., примеры: {s[bad].unique()[:5]}")
    if c in INT_COLS:
        if (df[c].dropna() % 1 == 0).all():
            df[c] = df[c].astype("Int64")
        else:
            print(f"{c}: есть дробные значения, оставлен float64")

print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов")
df.head(3)

### 1.2 Проверка и удаление полных дубликатов 

In [ ]:
# Полные дубликаты
dup_mask = df.duplicated(keep="first")
print("Полных дубликатов:", dup_mask.sum())
display(df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10))

n_before = len(df)
df = df[~dup_mask]
add_log("rent_v28*", "полные дубликаты строк", "вся выгрузка", int(dup_mask.sum()), f"{n_before} строк", "удалены, оставлен первый экземпляр", f"{len(df)} строк", "копия строки могла бы попасть и в train, и в test")
print(f"Строк: {n_before} -> {len(df)}")

### 1.3 Удаляем строки без целевой переменной

In [ ]:
# Строки без целевой переменной
empty_target = df["rent_rub"].isna()
print("Пустой rent_rub:", empty_target.sum())
display(df.loc[empty_target, ["listing_id", "rent_rub"]].head(10))

n_before = len(df)
df = df[~empty_target]

add_log("rent_rub", "пустое значение целевой переменной", "вся выгрузка", int(empty_target.sum()), f"{n_before} строк", "строки удалены", f"{len(df)} строк", "без цели нельзя ни обучить модель, ни посчитать метрики")
print(f"Строк: {n_before} -> {len(df)}")

Вывод: удалили все дубликаты и строки в которых не указана целевая переменная, теперь эти данные можно зарбивать на train, valid и est

## 2. Деление данных на test, validation и train
При разбиении выделим 15% на тест и 15% на валидацию, остальные 70% - train 

In [ ]:
from sklearn.model_selection import train_test_split

# Задаём параметры разбиения (по 15% на валиданию и тест)
RANDOM_STATE = 42
TEST_SIZE, VALID_SIZE = 0.15, 0.15
INTERIM_PATH = Path("..") / "data" / "interim" / "test.csv"

# Разбиваем целевую переменную на 10 квантилей для стратификации
price_bins = pd.qcut(df["rent_rub"].astype(float), q=10, duplicates="drop")
# отделяем тестовую выборку (15%) от остальной части данных с сохранением распределения цен
dev_idx, test_idx = train_test_split(df.index, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=price_bins)
# делим оставшиеся 85% на train и valid
train_idx, valid_idx = train_test_split(dev_idx, test_size=VALID_SIZE / (1 - TEST_SIZE), random_state=RANDOM_STATE, stratify=price_bins.loc[dev_idx])

# Сохраняем выбранные строки в тест в изначальном виде в отдельном файле
raw = pd.read_csv(RAW_PATH, sep="|", encoding="cp1251", dtype=str, keep_default_na=False, na_values=[""])
INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
raw.loc[test_idx.sort_values()].to_csv(INTERIM_PATH, sep="|", encoding="cp1251", index=False)

# Формируем итоговые датафреймы train, valid и test по полученным индексам
df_train, df_valid, df_test = df.loc[train_idx], df.loc[valid_idx], df.loc[test_idx]

# Выводим диагностику
test_check = pd.read_csv(INTERIM_PATH, sep="|", encoding="cp1251", dtype=str, keep_default_na=False, na_values=[""])
print(f"train {len(df_train)} | valid {len(df_valid)} | test {len(df_test)}")
print("Пересечений:", len(set(train_idx) & set(test_idx)), len(set(valid_idx) & set(test_idx)), len(set(train_idx) & set(valid_idx)))
print("Тест сохранён:", test_check.shape)

add_log("rent_v28*", "разбиение на выборки", "вся выгрузка", len(df), f"{len(df)} строк", f"train/valid/test, стратификация по децилям rent_rub, RANDOM_STATE={RANDOM_STATE}", f"{len(df_train)}/{len(df_valid)}/{len(df_test)}", "тест сохранён сырым в data/interim/, обучаемые параметры считаются только по train")

Сравниваем выборки между собой.
- числовые признаки: медиана, доля пропусков, статистика KS 
- категории: частоты и доли
- Цена(целевая): ЭФР

In [ ]:
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

# Сравниваем valid и test с train
parts = {"valid": df_valid, "test": df_test}

# Числовые признаки: сравниваем центр (медиану) и долю пропусков. (KS — максимальное расстояние между двумя ECDF(ЭФР) (долями наблюдений ≤ порога))
rows = []
for c in INT_COLS + FLOAT_COLS:
    a = df_train[c].dropna().astype(float)
    for name, part in parts.items():
        b = part[c].dropna().astype(float)
        ks = stats.ks_2samp(a, b)
        rows.append({"столбец": c, "выборка": name, "медиана train": a.median(), "медиана": b.median(), "KS": round(ks.statistic, 3), "p": round(ks.pvalue, 3), "пропуски train, %": round(df_train[c].isna().mean() * 100, 2), "пропуски, %": round(part[c].isna().mean() * 100, 2)})
num_cmp = pd.DataFrame(rows)
display(num_cmp.sort_values("KS", ascending=False).head(15))

# Категориальные признаки: сравниваем частоты и доли. Пропуск считаем отдельной категорией "<NaN>".
rows = []
for c in CAT_COLS:
    p_tr = df_train[c].astype("string").fillna("<NaN>").value_counts(normalize=True)
    for name, part in parts.items():
        p = part[c].astype("string").fillna("<NaN>").value_counts(normalize=True)
        diff = p_tr.sub(p, fill_value=0).abs() * 100
        rows.append({"столбец": c, "выборка": name, "макс. разница долей, п.п.": round(diff.max(), 2), "значение": diff.idxmax()})
display(pd.DataFrame(rows).sort_values("макс. разница долей, п.п.", ascending=False))

# Целевая переменная: ECDF(ЭФР) показывает долю объявлений с ценой ≤ порога 
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, part in {"train": df_train, "valid": df_valid, "test": df_test}.items():
    sns.ecdfplot(part["rent_rub"].astype(float), ax=ax[0], label=name)
    sns.kdeplot(np.log1p(part["rent_rub"].astype(float)), ax=ax[1], label=name)
ax[0].set_title("ECDF rent_rub"); ax[1].set_title("log1p(rent_rub), плотность")
ax[0].legend()
plt.show()

Вывод: разбиение случайно, видно, что KS и доли малы, значит выборки сопоставимы. Распределение целевой переменной на всех трёх кусках данных совпадает.

## 3. Аудит
Аудит проводится только на train: его результаты определяют правила, которые записываются в Pipeline
1. NumericCleaner - типы и невозможные значения
2. CategoryCleaner - единое написание категорий
3. ColumnTransformer - числа

In [ ]:
import sys
sys.path.insert(0, str(Path("..").resolve()))

TARGET = "rent_rub"  # целевая
ID_COLS = ["listing_id"]
EXCLUDE = ["response_time_hours"]  # не используем инфа после публикации(утечка)
NUM_FEATURES = [c for c in INT_COLS + FLOAT_COLS if c not in [TARGET] + EXCLUDE] # Все численные категории
CAT_FEATURES = [c for c in CAT_COLS if c not in EXCLUDE] # Все использумые категории
FEATURES = NUM_FEATURES + CAT_FEATURES

# x — признаки, y — цель
X_train, y_train = df_train[FEATURES], df_train[TARGET].astype(float)
X_valid, y_valid = df_valid[FEATURES], df_valid[TARGET].astype(float)
num_train = X_train[NUM_FEATURES].astype(float)
print("Признаков:", len(FEATURES), "| числовых:", len(NUM_FEATURES), "| категориальных:", len(CAT_FEATURES))

### 3.1 Паспорт данных

In [ ]:
passport = pd.DataFrame({
    "тип": df_train.dtypes.astype(str),
    "пропусков": df_train.isna().sum(),
    "доля пропусков, %": (df_train.isna().mean() * 100).round(2),
    "уникальных": df_train.nunique(dropna=True),
})
display(passport)

print("\nКатегориальные столбцы: сколько раз встречается каждое значение")
for col in CAT_COLS:
    vc = df_train[col].value_counts(dropna=False)
    print(f"\n{col} (категорий: {df_train[col].nunique()}):")
    display(vc.to_frame("количество").assign(доля=lambda t: (t["количество"] / len(df_train) * 100).round(2)))

### 3.2 Написание категорий

In [ ]:
from difflib import SequenceMatcher
from modules.preprocessing import normalize_text, canon_key

for c in CAT_FEATURES:
    norm = normalize_text(X_train[c])  # пробелы по краям, регистр, повторные пробелы
    key = canon_key(X_train[c])        # ключ: без знаков, ё=е, латиница -> кириллица
    counts = norm.value_counts()       # частоты нормализованных написаний
    print(f"\n{c}: значений {X_train[c].nunique()} -> после нормализации {norm.nunique()} -> по ключу {key.nunique()}")

    # Написания, которые отличаются только знаками, ё/е или латиницей: ключ у них общий
    groups = pd.DataFrame({"norm": norm, "key": key}).dropna().groupby("key")["norm"].agg(lambda v: sorted(v.unique()))
    same = groups[groups.map(len) > 1]
    if len(same):
        display(same.to_frame("написания с одним ключом"))

    # Похожие, но с разными ключами: вероятные опечатки (решение принимаем вручную)
    vals = list(counts.index)
    keymap = dict(zip(vals, canon_key(pd.Series(vals))))
    similar = [(a, b, round(SequenceMatcher(None, a, b).ratio(), 2)) for i, a in enumerate(vals) for b in vals[i + 1:] if keymap[a] != keymap[b] and SequenceMatcher(None, a, b).ratio() >= 0.8]
    if similar:
        display(pd.DataFrame(similar, columns=["написание 1", "написание 2", "сходство"]).assign(частота_1=lambda t: t["написание 1"].map(counts), частота_2=lambda t: t["написание 2"].map(counts)))

    # Полная таблица только для столбцов с большим числом значений
    if norm.nunique() > 6:
        display(counts.to_frame("количество").assign(доля=lambda t: (t["количество"] / len(df_train) * 100).round(2)))

#### 3.2.1 Объединение названий категорий

In [ ]:
from modules.preprocessing import CategoryCleaner

# Итоговые допустимые значения
VALID_CATEGORIES = {"renovation": ["евроремонт", "косметический", "без ремонта", "дизайнерский"]}

# Словари замен названий
CATEGORY_FIXES = {"renovation": {
    "косметика": "косметический",
    "косметический ремонт": "косметический",
    "дизайнерский ремонт": "дизайнерский",
    "нет ремонта": "без ремонта",     
    "требует ремонта": "без ремонта",
    "авторский": "дизайнерский",
}}

# Экземпляр "очистителя" обучаем на train
cleaner = CategoryCleaner(CAT_FEATURES, CATEGORY_FIXES, VALID_CATEGORIES).fit(X_train)
X_clean = cleaner.transform(X_train)  # новая таблица, X_train не изменился

display(X_clean["renovation"].value_counts(dropna=False).to_frame("количество"))
# Ищем знач не из VALID_CATEGORIES
left = X_clean["renovation"].value_counts().loc[lambda s: ~s.index.isin(VALID_CATEGORIES["renovation"])]
print("Неопознанные значения:"); display(left.to_frame("количество"))

add_log("renovation", "разные написания одной категории", "train, valid, test",
        int((X_train["renovation"].astype("string").str.strip().str.lower() != X_clean["renovation"]).sum()),
        "20 написаний", "нормализация + сопоставление с допустимыми + CATEGORY_FIXES",
        f"{X_clean['renovation'].nunique()} категорий", "OneHotEncoder не должен считать синонимы разными категориями")

### 3.3 Проверяем числовые признаки

In [ ]:
checks = pd.Series({
    "kitchen_area || living_area || total_area < 0" : (df_train["living_area"]<0).sum() + (df_train["kitchen_area"]<0).sum() + (df_train["total_area"]<0).sum(),
    "floor < 1": (df_train["floor"] < 1).sum(),
    "floors_total < 1": (df_train["floors_total"] < 1).sum(),
    "floor > floors_total": (df_train["floor"] > df_train["floors_total"]).sum(),
    "living_area > total_area": (df_train["living_area"] > df_train["total_area"]).sum(),
    "kitchen_area > total_area": (df_train["kitchen_area"] > df_train["total_area"]).sum(),
    "living + kitchen > total": ((df_train["living_area"] + df_train["kitchen_area"]) > df_train["total_area"]).sum(),
    "year_built < 1800": (df_train["year_built"] < 1800).sum(),
    "year_built > 2026": (df_train["year_built"] > 2026).sum(),
    "площадь <= 0": (df_train[["total_area", "living_area", "kitchen_area"]] <= 0).any(axis=1).sum(),
    "metro_distance_m < 0": (df_train["metro_distance_m"] < 0).sum(),
    "distance_to_center_km > 50": (df_train["distance_to_center_km"] > 50).sum(),
    "rent_rub <= 0": (df_train[TARGET] <= 0).sum(),
})
display(checks.to_frame("строк"))
print("Диапазон дат:", df_train["publication_date"].min(), "—", df_train["publication_date"].max())

display(df_train.loc[df_train["distance_to_center_km"] > 50, ["distance_to_center_km", "district", "metro_distance_m"]].head(10))
display(df_train.loc[df_train["floors_total"] < 1, ["floors_total", "floor", "house_type", "year_built"]].head(10))
display(df_train.loc[df_train["living_area"] > df_train["total_area"], ["living_area", "kitchen_area", "total_area"]].head(10))


NUM_BOUNDS = {"floor": (1, None), "year_built": (1800, 2026), "total_area": (1, None), "living_area": (1, None), "kitchen_area": (1, None), "ceiling_height": (2, 6), "metro_distance_m": (0, None), "distance_to_center_km": (0, None)}
NUM_PAIRS = [("floor", "floors_total"), ("living_area", "total_area"), ("kitchen_area", "total_area")]

#### 3.3.1 Числовая очистка

In [ ]:
from modules.preprocessing import NumericCleaner

# проверяем очистку на отдельной копии train
num_cleaner = NumericCleaner(NUM_FEATURES, NUM_BOUNDS, NUM_PAIRS)
X_num_clean = num_cleaner.fit_transform(X_train)

# Маски считаем по данным ДО очистки
meters_mask = X_train["distance_to_center_km"] > 50
zero_floors_mask = X_train["floors_total"] == 0

# Все расстояния, которые изменятся, и результат деления
display(pd.DataFrame({
        "было": X_train.loc[meters_mask, "distance_to_center_km"],
        "стало, км": X_num_clean.loc[meters_mask, "distance_to_center_km"],
    }).sort_values("было"))

# Проверяем новые пропуски в числовых признаках
new_na = X_train[NUM_FEATURES].notna() & X_num_clean[NUM_FEATURES].isna()
display(new_na.sum().loc[lambda s: s > 0].to_frame("новых пропусков"))

# Показываем площади после применения правил
area_mask = X_train["living_area"] > X_train["total_area"]
display(X_num_clean.loc[ area_mask, ["living_area", "kitchen_area", "total_area"]].head(10))

# объединяем обе очистки в одной таблице
X_clean = cleaner.transform(X_num_clean)

#### 3.3.2 Записываем решения в журнал

In [ ]:
# фиксируем принятое правило перевода единиц
add_log("distance_to_center_km", "предполагаемое смешение метров и километров", "train", int(meters_mask.sum()), "значения > 50", "деление на 1000 внутри NumericCleaner", "расстояние в километрах", "принята гипотеза: значения > 50 записаны в метрах; проверяется по таблице до/после")

# ноль -> неизвестная этажность
add_log("floors_total", "нулевая этажность", "train", int(zero_floors_mask.sum()), "0", "замена на NaN внутри NumericCleaner", "пропуск", "нулевая этажность трактуется как неизвестная; строки и остальные признаки сохраняются")

# фиксируем новые пропуски от остальных числовых правил
for c in NUM_FEATURES:
    mask = new_na[c].copy()
    # Нулевую этажность уже записали отдельно
    if c == "floors_total":
        mask &= ~zero_floors_mask.fillna(False)

    n = int(mask.sum())
    if n:
        add_log(c, "нарушение допустимого диапазона или отношения признаков", "train", n, str(X_train.loc[mask, c].head(5).tolist()), "замена на NaN внутри NumericCleaner",  "пропуск",  "правила NUM_BOUNDS и NUM_PAIRS; в паре сохраняем опорное поле, недопустимое зависимое значение считаем неизвестным")

# отдельная неоднозначность: пока без автоматической замены
area_sum_mask = ((X_train["living_area"] <= X_train["total_area"])
               & (X_train["kitchen_area"] <= X_train["total_area"])
               & (X_train["living_area"] + X_train["kitchen_area"]> X_train["total_area"]))

add_log("living_area + kitchen_area", "сумма площадей больше общей после остальных проверок", "train", int(area_sum_mask.sum()),  "living_area + kitchen_area > total_area", "living_area и kitchen_area заменены на NaN", "общая площадь сохранена", "невозможно надёжно определить ошибочную составляющую; обе составляющие считаем неизвестными")


### 3.4 Проверяем результаты

In [ ]:
# общий результат очистки
missing = pd.DataFrame({
    "пропусков до": X_train.isna().sum(),
    "новых пропусков": (X_train.notna() & X_clean.isna()).sum(),
    "пропусков после": X_clean.isna().sum(),
})
display(missing)

checks_after = pd.Series({
    "floors_total == 0": (X_clean["floors_total"] == 0).sum(),
    "floor > floors_total": (X_clean["floor"] > X_clean["floors_total"]).sum(),
    "living_area > total_area": (X_clean["living_area"] > X_clean["total_area"]).sum(),
    "kitchen_area > total_area": (X_clean["kitchen_area"] > X_clean["total_area"]).sum(),
    "living + kitchen > total": ((X_clean["living_area"] + X_clean["kitchen_area"]) > X_clean["total_area"]).sum(),
    "distance_to_center_km > 50": (X_clean["distance_to_center_km"] > 50).sum(),
})
display(checks_after.to_frame("нарушений после очистки"))
print("Строк до / после:", len(X_train), "/", len(X_clean))
# Если один из участников пары неизвестен, сравнить их нельзя
print("Неизвестная этажность:", X_clean["floors_total"].isna().sum())
print("Неизвестная жилая площадь:", X_clean["living_area"].isna().sum())

display(X_train.loc[X_clean["living_area"]+X_clean["kitchen_area"]  > X_clean["total_area"], ["living_area", "kitchen_area", "total_area"]].head(10))

### 3.5 Распределения числовых признаков


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Только известные значения после очистки (пропуски пока не заполняем)
num_clean = X_clean[NUM_FEATURES].astype(float)

# Таблица: сколько значений известно и как устроено распределение
dist = pd.DataFrame({
    "известно": num_clean.notna().sum(),
    "min": num_clean.min(),
    "25%": num_clean.quantile(0.25),
    "медиана": num_clean.median(),
    "75%": num_clean.quantile(0.75),
    "max": num_clean.max(),
    "асимметрия": num_clean.skew(),   # > 0 — правый хвост, < 0 — левый
})
display(dist.round(2))

# Признаки с малым числом значений рисуем столбцами по значениям
DISCRETE = ["rooms", "max_tenants"]

n_cols = 3
n_rows = int(np.ceil(len(NUM_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3.2 * n_rows))
axes = axes.ravel()

for ax, c in zip(axes, NUM_FEATURES):
    data = num_clean[c].dropna()
    sns.histplot(data, bins=30, discrete=(c in DISCRETE), ax=ax)
    ax.axvline(data.median(), color="red", linestyle="--", linewidth=1)  # красная линия — медиана
    ax.set_title(f"{c} (известно {len(data)})")
    ax.set_xlabel("")

# Пустые ячейки сетки скрываем
for ax in axes[len(NUM_FEATURES):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

### 3.6 Распределение целевой переменной и её связь с остальными числовыми признаками

In [ ]:
# анализируем уже очищенные числа
num_train = X_clean[NUM_FEATURES].astype(float)
display(num_train.corrwith(y_train, method="spearman").sort_values(key=abs, ascending=False).round(3).to_frame("Spearman с rent_rub"))
display(df_train.groupby("district", observed=True)[TARGET].agg(["count", "median"]).sort_values("median"))

import seaborn as sns
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y_train, bins=50, ax=ax[0])
sns.histplot(np.log1p(y_train), bins=50, ax=ax[1])
ax[0].set_title("rent_rub")
ax[1].set_title("log1p(rent_rub)")
plt.show()

## 4. Анализ и обработка пропусков
### 4.1 Подготовка данных для анализа пропусков
Применяем только категориальную и числовую очистку

In [ ]:
from modules.preprocessing import NumericCleaner, CategoryCleaner

# Параметры и правила берём из главы 3
num_cleaner = NumericCleaner(NUM_FEATURES, NUM_BOUNDS, NUM_PAIRS)
cleaner = CategoryCleaner(CAT_FEATURES, CATEGORY_FIXES, VALID_CATEGORIES)

X_num_clean = num_cleaner.fit_transform(X_train)
X_clean = cleaner.fit_transform(X_num_clean)

# Validation: только transform, без обучения новых соответствий
X_valid_clean = num_cleaner.transform(X_valid)
X_valid_clean = cleaner.transform(X_valid_clean)

print("Строк train:", len(X_clean))
print("Строк valid:", len(X_valid_clean))

### 4.2 Сколько пропусков и происхождение

In [ ]:
missing = pd.DataFrame({
    "до очистки": X_train.isna().sum(),
    "добавлено очисткой": (X_train.notna() & X_clean.isna()).sum(),
    "после очистки": X_clean.isna().sum(),
    "после, %": (X_clean.isna().mean() * 100).round(2),
})
display(missing.sort_values("после очистки", ascending=False))

print("Строк хотя бы с одним пропуском:", X_clean.isna().any(axis=1).sum())
print("Полных строк:", X_clean.notna().all(axis=1).sum())
print("Потеряли бы при dropna, %:", round(X_clean.isna().any(axis=1).mean() * 100, 2))

# Проверяем, нет ли столбца, который полностью пуст
print("Полностью пустые столбцы:", X_clean.columns[X_clean.isna().all()].tolist())

# Сколько пропусков приходится на одну строку
display(X_clean.isna().sum(axis=1).value_counts().sort_index().to_frame("число объявлений"))


Вывод: если удалять все строки с пропусками, то потеряем 40% выборки - не подходит. Нужно изучить: есть ли закономерности(связь) между признаками и целевой переменной

## 4.3 Пропуски и стоимость аренды

In [ ]:
missing_cols = missing.index[missing["после очистки"] > 0].tolist()

for c in missing_cols:
    temp = pd.DataFrame({ "есть пропуск": X_clean[c].isna(), "rent_rub": y_train,})
    
    # Считаем метрики
    stats = temp.groupby("есть пропуск")["rent_rub"].agg(["count", "median", "mean"]).round(2)
    quantiles = temp.groupby("есть пропуск")["rent_rub"].quantile([0.25, 0.75]).unstack().round(2)
    
    # Объединяем в одну таблицу
    combined_table = pd.concat([stats, quantiles], axis=1)
    
    print(f"\n{c} | False — известно, True — пропуск")
    display(combined_table)

Вывод: самая сильная закономерность в площади кухни, если не указана площадь кухни цена сильно снижается(по медиане) относительно квартир, где площадь кухни указана.

## 4.4 Связи числовых признаков

In [ ]:
# Связи с целевой переменной и между признаками
num_train = X_clean[NUM_FEATURES].astype(float)

display(num_train.corrwith(y_train, method="spearman")
        .sort_values(key=abs, ascending=False).round(3)
        .to_frame("Spearman с rent_rub"))

corr = num_train.corr(method="spearman")
rows = []
for i, a in enumerate(NUM_FEATURES):
    for b in NUM_FEATURES[i + 1:]:
        if abs(corr.loc[a, b]) >= 0.8:
            rows.append({"признак 1": a, "признак 2": b, "Spearman": round(corr.loc[a, b], 3), "известны оба": int(num_train[[a, b]].notna().all(axis=1).sum()),})
display(pd.DataFrame(rows, columns=["признак 1", "признак 2", "Spearman", "известны оба"]))

Возможно отсутствие площади кухни может говорить об типе квартир, в которых отсутствует отдельно кухня. 

### 4.5 Анализ исходных пропусков кухни

In [ ]:
# Маски строим до заполнения
kitchen_original_missing = X_train["kitchen_area"].isna()
kitchen_added_missing = X_train["kitchen_area"].notna() & X_clean["kitchen_area"].isna()

print("Исходных пропусков кухни:", kitchen_original_missing.sum())
print("Добавлено очисткой:", kitchen_added_missing.sum())

# Гипотезу об исходных пропусках проверяем отдельно
kitchen_analysis = pd.DataFrame({
    "rooms": X_clean["rooms"],
    "исходный пропуск кухни": kitchen_original_missing,
    "total_area": X_clean["total_area"],
})
display(kitchen_analysis.groupby("rooms").agg(объявлений=("исходный пропуск кухни", "size"), исходных_пропусков=("исходный пропуск кухни", "sum"), доля_пропусков=("исходный пропуск кухни", "mean")).assign(доля_пропусков=lambda t: (t["доля_пропусков"] * 100).round(2)))

# Сравниваем три группы, не смешивая причины неизвестности
status = pd.Series("известна после очистки", index=X_clean.index)
status.loc[kitchen_original_missing] = "исходный пропуск"
status.loc[kitchen_added_missing] = "добавлено очисткой"
kitchen_groups = pd.DataFrame({"статус кухни": status, "total_area": X_clean["total_area"]})
display(kitchen_groups.groupby("статус кухни")["total_area"].agg(["count", "median", "mean"]).round(2))

Вывод: неуказана площадь кухни только в квартирах с 0 комнат, из чего можно сдделать вывод, что это обозначение квартир-студий, где нет как таковых отдельных комнат и соответственно кухни.
Решение: гипотеза: отсутствие знач площади кухни == студия
- площадь кухни ставим ноль и сохраняем флаг
- остальные неизвестные площади кухни и жилой зоны оцениваем по медианным долям от общей площади

### 4.6 Pipeline заполнения площадей

In [ ]:
from sklearn.pipeline import Pipeline
from modules.preprocessing import NumericCleaner, CategoryCleaner, KitchenMissingFlag, AreaShareImputer

# Цепочка очистки и заполнения площадей
area_pipeline = Pipeline([
    ("kitchen_flag", KitchenMissingFlag()),
    ("numeric_cleaner", NumericCleaner(NUM_FEATURES, NUM_BOUNDS, NUM_PAIRS)),
    ("category_cleaner", CategoryCleaner(CAT_FEATURES, CATEGORY_FIXES, VALID_CATEGORIES)),
    ("area_imputer", AreaShareImputer()),
])

# Обучаем только на train
X_area_filled = area_pipeline.fit_transform(X_train)
X_valid_area_filled = area_pipeline.transform(X_valid)

imputer = area_pipeline.named_steps["area_imputer"]
print("Медианная доля жилой площади:", round(imputer.living_share_, 3))
print("Медианная доля кухни:", round(imputer.kitchen_share_, 3))

4.6.1 Проверка результата

In [ ]:
# Состояние до заполнения площадей
X_before_area = area_pipeline[:-1].transform(X_train)
X_valid_before_area = area_pipeline[:-1].transform(X_valid)

original = X_before_area["kitchen_area_original_missing"].eq(1)
added = X_before_area["kitchen_area"].isna() & ~original
living_missing = X_before_area["living_area"].isna()

# Сводка по пропускам и заменам
checks = pd.Series({
    "исходных пропусков кухни": original.sum(),
    "из них заменено на 0": X_area_filled.loc[original, "kitchen_area"].eq(0).sum(),
    "добавлено пропусков кухни очисткой": added.sum(),
    "из них заполнено оценкой": X_area_filled.loc[added, "kitchen_area"].notna().sum(),
    "пропусков жилой до заполнения": living_missing.sum(),
    "из них заполнено оценкой жилой": X_area_filled.loc[living_missing, "living_area"].notna().sum(),
})
display(checks.to_frame("строк"))

# Пропуски до и после заполнения
cols = ["total_area", "living_area", "kitchen_area"]
display(pd.DataFrame({
    "train до": X_before_area[cols].isna().sum(),
    "train после": X_area_filled[cols].isna().sum(),
    "valid до": X_valid_before_area[cols].isna().sum(),
    "valid после": X_valid_area_filled[cols].isna().sum(),
}))

# Проверка конфликтов площадей (living + kitchen > total)
for name, before, after in [
    ("train", X_before_area, X_area_filled),
    ("valid", X_valid_before_area, X_valid_area_filled),
]:
    conflict_before = (before["living_area"] + before["kitchen_area"]) > before["total_area"]
    conflict_after = (after["living_area"] + after["kitchen_area"]) > after["total_area"]
    new_conflicts = (conflict_after & ~conflict_before).sum()
    print(f"{name}: было противоречий — {conflict_before.sum()}, новых — {new_conflicts}")

In [ ]:
# Используем уже полученные промежуточные и итоговые таблицы
area_parts = [("train", X_before_area, X_area_filled), ("valid", X_valid_before_area, X_valid_area_filled)]
area_imputer = area_pipeline.named_steps["area_imputer"]

for scope, before, after in area_parts:
    original = before["kitchen_area_original_missing"].eq(1)
    kitchen_missing = before["kitchen_area"].isna() & ~original
    living_missing = before["living_area"].isna()

    # Считаем только реально выполненные замены
    zero_filled = original & after["kitchen_area"].eq(0)
    kitchen_filled = kitchen_missing & after["kitchen_area"].notna()
    living_filled = living_missing & after["living_area"].notna()

    add_log("kitchen_area", "исходный пропуск площади кухни", scope, int(zero_filled.sum()), 
            f"исходных пропусков: {int(original.sum())}", "технический 0 после NumericCleaner; сохранён флаг исходного пропуска", f"нулей подставлено: {int(zero_filled.sum())}", "ноль кодирует исходную неизвестность и не доказывает отсутствие кухонной зоны")

    add_log("kitchen_area", "пропуск площади кухни, созданный очисткой", scope, int(kitchen_filled.sum()), 
            f"пропусков до заполнения: {int(kitchen_missing.sum())}", "оценка по train-доле от общей площади с ограничением доступным остатком", f"заполнено: {int(kitchen_filled.sum())}; осталось: {int((kitchen_missing & after['kitchen_area'].isna()).sum())}", f"медианная доля train={area_imputer.kitchen_share_:.4f}; запас при ограничении 0.95; известные площади сохранены")

    add_log("living_area", "исходный или созданный очисткой пропуск жилой площади", scope, int(living_filled.sum()), 
            f"пропусков до заполнения: {int(living_missing.sum())}", "оценка по train-доле от общей площади с ограничением доступным остатком", f"заполнено: {int(living_filled.sum())}; осталось: {int((living_missing & after['living_area'].isna()).sum())}", f"медианная доля train={area_imputer.living_share_:.4f}; запас при ограничении 0.95; rent_rub не используется")

    # Одна строка может иметь сразу несколько флагов
    flags = ["kitchen_area_original_missing", "living_area_missing", "kitchen_area_cleaning_missing"]
    flagged_rows = after[flags].eq(1).any(axis=1)

    add_log("area_missing_flags", "необходимо сохранить происхождение заполненных площадей", scope, int(flagged_rows.sum()), "исходные пропуски и пропуски после очистки", "созданы три бинарных флага до заполнения соответствующих площадей", ", ".join(flags), "флаги различают исходную неизвестность кухни и неизвестные после очистки площади")

### 4.7 Признаки 

In [ ]:
# Список для числовой ветки
AREA_FLAGS = ["kitchen_area_original_missing", "living_area_missing", "kitchen_area_cleaning_missing"]
NUM_FEATURES_FILLED = NUM_FEATURES + AREA_FLAGS